# Machine Learning for Hydrologic Parameter Estimation

This notebook details the final stage of our project: the development, evaluation, and selection of machine learning models to predict the ModClark hydrologic parameters, Time of Concentration ($T_c$) and Storage Coefficient ($R$), from watershed geomorphological and land use characteristics.

Building upon the Exploratory Data Analysis (EDA) conducted previously ([see notebook here](15_exploratory_data_analysis.ipynb)), this notebook implements the machine learning workflow. We will train and compare a suite of models, ranging from interpretable linear regressions to powerful non-linear ensembles. The ultimate goal is to identify the best-performing model that can accurately and reliably estimate $T_c$ and $R$ for ungauged basins, providing a valuable tool for hydrological modeling.

## Modeling Strategy

The insights from our EDA directly inform a clear and targeted modeling strategy. The key decisions are outlined below:

* **Target Variable Transformation:** The EDA revealed that the target variables, $T_c$ and $R$, are right-skewed and their relationships with many features are non-linear. Therefore, to improve model performance and better satisfy the assumptions of linear models, we will model the **natural logarithm** of the targets: **$\log(T_c)$** and **$\log(R)$**.

* **Tailored Feature Preprocessing:** Our EDA showed that linear models require feature transformations to meet linearity assumptions, while non-linear models do not. Therefore, we will create two separate preprocessing pipelines:
    1.  **Pipeline for Linear Models (OLS, Elastic Net):** This pipeline will include the **feature transformations** (Log/Box-Cox) suggested by the bivariate analysis, followed by **Z-score standardization**.
    2.  **Pipeline for Non-Linear Models (Random Forest, SVM):** This pipeline will only perform **Z-score standardization**. While not strictly necessary for Random Forests, it is essential for SVM and ensures consistency.

* **Feature Selection Strategy:** To create more parsimonious and interpretable models, we will incorporate feature selection:
    * **For Linear Models:** For standard linear regression, we will drop small coefficients. For elastic net regression we will leverage the **L1 regularization component of Elastic Net** as our primary method for feature selection. By tuning its strength, the model will automatically drive the coefficients of less important or redundant features to zero, effectively selecting a simpler feature subset.
    * **For Non-Linear Models:** While models like Random Forest are robust to irrelevant features, feature selection can still improve performance and interpretability. We will use the **feature importances generated by an initial Random Forest** run to guide the selection of a more focused feature set if necessary.

## Model Training and Evaluation Workflow

We will follow a, multi-step process to ensure our model evaluation is unbiased and reliable. Two separate modeling pipelines will be developed, one for predicting $\log(T_c)$ and one for $\log(R)$.

1.  **Data Splitting:** The dataset will first be split into a **training/validation pool (80% of watersheds)** and a final **hold-out test set (20% of watersheds)**. This split will be performed based on `ws_id` to ensure that all data points from a single watershed reside in only one set, preventing data leakage.

2.  **Pipeline Construction:** For each model, we will use `scikit-learn`'s `Pipeline` object, incorporating the appropriate preprocessing steps (Transformation + Scaling for linear, Scaling only for non-linear).

3.  **Model Selection via Cross-Validation:** We will use **Group K-Fold Cross-Validation** on the training/validation pool, using `ws_id` as the grouping variable. Each of our candidate models—**Multiple Linear Regression, Elastic Net, SVM, and Random Forest**—will be trained and evaluated across all folds. This process will allow us to reliably compare their performance.

4.  **Final Model Training:** After identifying the best-performing model from cross-validation, it will be re-trained one last time on the **entire training/validation pool**.

5.  **Final Evaluation:** This final, retrained model will be evaluated on the **untouched hold-out test set** to provide the definitive, unbiased assessment of its ability to generalize to new, unseen watersheds.

### Performance Metrics
The models will be compared using a standard set of regression and hydrological metrics, including:
* **Root Mean Squared Error (RMSE)**
* **Nash-Sutcliffe Efficiency (NSE)**
* **Mean Absolute Percentage Error (MAPE) of Peak Flow**
* **Bias of Peak Flow**
* **Mean Absolute Error (MAE) of Time to Peak**
* **Bias of Total Volume**

In [1]:
# Set root folder for the project
from pathlib import Path

# Custom Modules
PROJECT_ROOT = Path.cwd().parent.parent

In [6]:
# Load dataframes
import pandas as pd
df_master_path = PROJECT_ROOT / "data/gold/tabular/machine_learning/df_master.csv"
df_recommendations_path = PROJECT_ROOT / "data/gold/tabular/machine_learning/feature_transformation_recommendations.csv"

df_master = pd.read_csv(df_master_path, dtype={'ws_id':str})
df_recommendations = pd.read_csv(df_recommendations_path)

### 1. Data Splitting

In [3]:
import pandas as pd
import numpy as np

# --- Assume df_master is a pre-existing pandas DataFrame ---
# It must contain a 'ws_id' column.

#-------------------------------------------------------------------------------
## 1. Data Splitting (Group-Based)
#-------------------------------------------------------------------------------

print("--- Splitting data into Training Pool and Hold-Out Test Set ---")

# 1. Get a list of unique watershed IDs
unique_ws_ids = df_master['ws_id'].unique()
np.random.seed(42) # for reproducibility
np.random.shuffle(unique_ws_ids)

# 2. Split the list of watershed IDs into 80% and 20%
split_index = int(len(unique_ws_ids) * 0.8)
train_pool_ws_ids = unique_ws_ids[:split_index]
test_set_ws_ids = unique_ws_ids[split_index:]

# 3. Create the dataframes based on the split IDs
df_train_pool = df_master[df_master['ws_id'].isin(train_pool_ws_ids)].copy()
df_test = df_master[df_master['ws_id'].isin(test_set_ws_ids)].copy()


# --- Summary of the Split ---
print("\n--- Split Summary ---")
print(f"Total unique watersheds: {len(unique_ws_ids)}")
print(f"Watersheds in Training/Validation Pool: {len(train_pool_ws_ids)} (~80%)")
print(f"Watersheds in Hold-Out Test Set: {len(test_set_ws_ids)} (~20%)")
print("-" * 30)
print(f"Total data points: {len(df_master)}")
print(f"Data points in Training/Validation Pool: {len(df_train_pool)}")
print(f"Data points in Hold-Out Test Set: {len(df_test)}")

# Verify that there are no overlapping watersheds between the two sets
assert len(np.intersect1d(train_pool_ws_ids, test_set_ws_ids)) == 0
print("\nVerification successful: No watershed overlap between sets. ✅")

--- Splitting data into Training Pool and Hold-Out Test Set ---

--- Split Summary ---
Total unique watersheds: 112
Watersheds in Training/Validation Pool: 89 (~80%)
Watersheds in Hold-Out Test Set: 23 (~20%)
------------------------------
Total data points: 292
Data points in Training/Validation Pool: 226
Data points in Hold-Out Test Set: 66

Verification successful: No watershed overlap between sets. ✅


### 2. Pipeline Constructor

In [32]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer, FunctionTransformer
from sklearn.compose import ColumnTransformer

# --- Assume df_master, features_to_analyze, and df_recommendations are pre-existing ---

#-------------------------------------------------------------------------------
## 2. Pipeline Construction (Dynamic and Robust to Zeros)
#-------------------------------------------------------------------------------

# --- New: Define a safe log transform function ---
def log_transform_safe(x):
    """Adds a small constant to handle log(0) issues."""
    epsilon = 1e-6
    return np.log(x + epsilon)

print("--- Building Dynamic Preprocessing Pipelines Based on Recommendations ---")

def create_preprocessor_for_target(target_name, recommendations_df, feature_list):
    """Builds a preprocessing pipeline for a specific target variable."""

    # Filter recommendations for the given target
    rec_target = recommendations_df[recommendations_df['Target'] == target_name]

    # Create lists of features for each transformation type
    log_features = rec_target[rec_target['Best_Transform_Type'] == 'Log(x)']['Feature'].tolist()
    boxcox_features = rec_target[rec_target['Best_Transform_Type'] == 'Box-Cox(x)']['Feature'].tolist()
    
    # Filter to only features present in the final dataframe
    log_features = [f for f in log_features if f in feature_list]
    boxcox_features = [f for f in boxcox_features if f in feature_list]

    # Define the transformer steps
    transformer_steps = []
    if log_features:
        # Use the new safe log transform function here
        transformer_steps.append(
            ('log', FunctionTransformer(log_transform_safe), log_features)
        )
    if boxcox_features:
        transformer_steps.append(
            ('power', PowerTransformer(method='box-cox'), boxcox_features)
        )

    # Create the ColumnTransformer
    feature_transformer = ColumnTransformer(
        transformers=transformer_steps,
        remainder='passthrough'
    )

    # Create the full pipeline
    preprocessor_pipeline = Pipeline(steps=[
        ('feature_transformer', feature_transformer),
        ('scaler', StandardScaler())
    ])
    
    return preprocessor_pipeline

# --- Create the specific pipelines for TC and R ---
col_names = df_master.columns.tolist()
remove_cols = ['ws_id','year','tc','r']
features_to_analyze = [f for f in col_names if f not in remove_cols]
linear_preprocessor_tc = create_preprocessor_for_target('TC', df_recommendations, features_to_analyze)
linear_preprocessor_r = create_preprocessor_for_target('R', df_recommendations, features_to_analyze)


# --- Summary of Pipelines ---
print("\n✅ Pipeline for Target 'TC' ('preprocessor_tc') created:")
print(linear_preprocessor_tc)

print("\n✅ Pipeline for Target 'R' ('preprocessor_r') created:")
print(linear_preprocessor_r)

#-------------------------------------------------------------------------------
## 2.2 Pipeline for Non-Linear Models (Random Forest, SVM)
#-------------------------------------------------------------------------------

# This pipeline only scales the features, as transformations for linearity
# are not required for these models. Scaling is still essential for SVM.
nonlinear_preprocessor = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# --- Summary of Pipeline ---
print("✅ Pipeline for Non-Linear Models ('nonlinear_preprocessor') created:")
print(nonlinear_preprocessor)

--- Building Dynamic Preprocessing Pipelines Based on Recommendations ---

✅ Pipeline for Target 'TC' ('preprocessor_tc') created:
Pipeline(steps=[('feature_transformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('log',
                                                  FunctionTransformer(func=<function log_transform_safe at 0x1384a2ac0>),
                                                  ['sinuosity', 'pct_barren']),
                                                 ('power',
                                                  PowerTransformer(method='box-cox'),
                                                  ['basin_area',
                                                   'centroidal_flowpath',
                                                   'main_channel_slope',
                                                   'flowpath_1085_slope',
                                                   'compactness_coefficient',
 

### 3. Model e Evaluation
#### 3.1 Multiple Linear Regression